# 13 - Meta-ensemble con búsqueda de pesos y umbral

Este notebook combina varios modelos fuertes (SVM/ET/RF/HGB + boosting opcional) optimizando pesos de blending para subir f1-macro.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for c in candidates:
        if (c / 'materials' / 'dataset_task3_exist2026').exists() and (c / 'notebooks_shiyi').exists():
            return c / 'notebooks_shiyi'
    if (cwd / 'artifacts').exists() and (cwd / 'entregables').exists():
        return cwd
    raise FileNotFoundError('No se pudo resolver notebooks_shiyi.')

PROJECT_ROOT = resolve_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
PRED_DIR = PROJECT_ROOT / 'predicciones'
REPORTS_DIR = PROJECT_ROOT / 'reports_hpo'
for d in [PRED_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

bundle = np.load(ARTIFACTS_DIR / 'fusion_features.npz', allow_pickle=True)
X_all = bundle['X_fusion'].astype(np.float32)
y_all = bundle['y'].astype(np.int64)
langs_all = bundle['langs'].astype(str)
sample_ids = bundle['sample_ids'].astype(str)

print('X_all:', X_all.shape, '| etiquetas validas:', int((y_all >= 0).sum()))

In [ ]:
def get_base_models():
    models = {
        'svm': Pipeline([('scaler', StandardScaler()), ('clf', SVC(C=10.0, gamma='scale', probability=True, class_weight='balanced', random_state=SEED))]),
        'et': ExtraTreesClassifier(n_estimators=700, max_depth=None, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'rf': RandomForestClassifier(n_estimators=700, class_weight='balanced_subsample', random_state=SEED, n_jobs=-1),
        'hgb': HistGradientBoostingClassifier(learning_rate=0.05, max_depth=12, max_leaf_nodes=63, random_state=SEED)
    }

    try:
        from xgboost import XGBClassifier
        models['xgb'] = XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            n_estimators=900,
            max_depth=8,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.8,
            random_state=SEED,
            n_jobs=-1
        )
    except Exception:
        pass

    return models

def random_weight_search(y_true, prob_matrix, n_trials=4000):
    n_models = prob_matrix.shape[1]
    best = {'f1': -1.0, 'w': None, 't': 0.5}

    for _ in range(n_trials):
        w = np.random.dirichlet(np.ones(n_models))
        p = prob_matrix @ w
        for t in np.linspace(0.2, 0.8, 121):
            pred = (p >= t).astype(int)
            f1 = f1_score(y_true, pred, average='macro')
            if f1 > best['f1']:
                best = {'f1': float(f1), 'w': w.copy(), 't': float(t)}

    return best

In [ ]:
def export_json(sample_ids, pred_binary, output_path):
    rows = [
        {'test_case': 'EXIST2026', 'id': str(sid), 'value': 'YES' if int(v) == 1 else 'NO'}
        for sid, v in zip(sample_ids, pred_binary)
    ]
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False)

def run_config(tag, train_langs):
    mask = np.isin(langs_all, train_langs) & (y_all >= 0)
    Xtr = X_all[mask]
    ytr = y_all[mask]

    if len(np.unique(ytr)) < 2:
        raise RuntimeError(f'{tag}: subset sin dos clases.')

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    models = get_base_models()

    prob_oof = []
    model_names = []
    fitted = {}

    for name, model in models.items():
        p = cross_val_predict(model, Xtr, ytr, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
        prob_oof.append(p)
        model_names.append(name)
        model.fit(Xtr, ytr)
        fitted[name] = model

    M = np.vstack(prob_oof).T
    best = random_weight_search(ytr, M, n_trials=5000)

    w_map = {n: float(w) for n, w in zip(model_names, best['w'])}
    print(tag, 'best_f1=', best['f1'], 'thr=', best['t'])
    print('weights:', w_map)

    p_all_list = []
    for n in model_names:
        p_all_list.append(fitted[n].predict_proba(X_all)[:, 1])
    p_all = np.vstack(p_all_list).T @ best['w']
    pred_all = (p_all >= best['t']).astype(int)

    pred_train = ((M @ best['w']) >= best['t']).astype(int)
    print(classification_report(ytr, pred_train, digits=4))

    rep = pd.DataFrame([{'config': tag, 'best_f1_macro': best['f1'], 'threshold': best['t'], **w_map}])
    rep.to_csv(REPORTS_DIR / f'meta_ensemble_{tag}.csv', index=False)

    out_path = PRED_DIR / f'BeingChillingWeWillWin_META_{tag}.json'
    export_json(sample_ids, pred_all, out_path)

run_config('ES', ['es'])
run_config('EN', ['en'])
run_config('ES_EN', ['es', 'en'])